# LabConstrictor Playground
---
Check that this computer, this installation and (if you have one) its GPU are in order, and try every feature of the LabConstrictor tools in Napari, Fiji and QuPath. The same checks are available there as tools: *Check everything*, *List the devices*, *Run a small test on a device*, *Make test data* and a set of *Stress* tools.

# Step 0: Initialize environment & install dependencies
---

In [ ]:
# @title Initialize environment & install dependencies
import sys

# Metadata
current_version = "0.1.0"
notebook_name = "LabConstrictorPlayground"

if "google.colab" in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")
    !pip install -q "labconstrictor-playground @ git+https://github.com/CellMigrationLab/LabConstrictor-Playground"
    !pip install -q https://github.com/CellMigrationLab/LabConstrictor-Tools/archive/refs/heads/main.zip
    print("✅ Colab setup done")
else:
    print("✅ Local environment detected. Assuming dependencies are already installed.")

import ipywidgets as widgets
from IPython.display import Image, Markdown, clear_output, display

import labconstrictor_playground
from labconstrictor_playground import check_everything, make_test_data, write_report
from labconstrictor_tools import diagnostics

print("%s %s, labconstrictor_playground %s" % (notebook_name, current_version, labconstrictor_playground.__version__))


# Step 1: Check everything
---
Tests the machine (memory, disk, write access), the worker, the GPU tools and PyTorch (CUDA, Apple Metal, ROCm), with a small speed test. Every problem comes with what to do about it. Attach the saved `report.json` when you report a problem.

In [ ]:
# @title Check everything
import pandas as pd
from pathlib import Path

speed = widgets.Checkbox(value=True, description="Run the speed test (CPU against every GPU)")
internet = widgets.Checkbox(value=False, description="Also try the internet (pypi.org, huggingface.co, github.com)")
run = widgets.Button(description="Check everything", button_style="primary")
out = widgets.Output()


def on_run(_):
    with out:
        clear_output(wait=True)
        print("Checking...")
        checks = check_everything(benchmark=speed.value, network=internet.value)
        files = write_report(checks, Path.home() / "LabConstrictorPlayground" / "reports")
        clear_output(wait=True)
        display(Markdown(diagnostics.summary(checks)))
        display(pd.DataFrame(diagnostics.rows(checks)))
        if "figure" in files:
            display(Image(filename=str(files["figure"])))
        print("Report saved in", files["json"].parent, "(attach report.json to an issue)")


run.on_click(on_run)
display(widgets.VBox([speed, internet, run, out]))


# Step 2: Make test data
---
Small images the tools can be tried on: a 3-channel image whose channels have the means 10, 20 and 30, an RGB image, labels, a time-lapse and a table of points.

In [ ]:
# @title Make test data
import tifffile
import matplotlib.pyplot as plt
from pathlib import Path

folder = widgets.Text(value=str(Path.home() / "LabConstrictorPlayground" / "test_data"), description="Folder", layout=widgets.Layout(width="70%"))
kind = widgets.Dropdown(options=["all", "three_channels", "rgb", "labels", "timelapse", "points"], value="all", description="Kind")
make = widgets.Button(description="Make test data", button_style="primary")
out2 = widgets.Output()


def on_make(_):
    with out2:
        clear_output(wait=True)
        written = make_test_data(folder.value, kind.value)
        for name, path in written.items():
            print("%-15s %s" % (name, path))
        shown = [p for n, p in written.items() if p.suffix == ".tif"][:3]
        if shown:
            fig, axes = plt.subplots(1, len(shown), figsize=(4 * len(shown), 4), squeeze=False)
            for ax, path in zip(axes[0], shown):
                array = tifffile.imread(path)
                while array.ndim > 2 and not (array.ndim == 3 and array.shape[-1] == 3):
                    array = array[0]
                ax.imshow(array, cmap="gray")
                ax.set_title(path.name)
                ax.axis("off")
            plt.show()


make.on_click(on_make)
display(widgets.VBox([folder, kind, make, out2]))
